# ADL Results Explorer

Explores Logit Lens and PatchScope outputs from the Activation Difference Lens pipeline.

In [1]:
from pathlib import Path
import matplotlib.pyplot as plt

# --- Configuration (edit these) ---
RESULTS_DIR = Path(
    "/workspace/model-organisms/diffing_results/olmo2_1B/milsub_wide-dpo/activation_difference_lens"
)
LAYERS = [7, 14, 15]
DATASET = "tulu-3-sft-olmo-2-mixture"
LOGIT_LENS_POSITION = -1  # Position for per-position logit lens view
PATCHSCOPE_POSITION = -1  # Position for per-position patchscope view
N_POSITIONS = 128  # Total positions (config: n)
LOGIT_LENS_MAX_ROWS = 20  # Set to an integer to truncate logit lens tables
PATCHSCOPE_GRADER = "openai_gpt-5-mini"
MODEL_ID = "allenai/OLMo-2-0425-1B-DPO"

LAYER_DIRS = {layer: RESULTS_DIR / f"layer_{layer}" / DATASET for layer in LAYERS}

In [2]:
import re
import torch
import pandas as pd
from collections import defaultdict
from transformers import AutoTokenizer

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 60)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)


def fmt_prob(p):
    """Format probability: scientific notation for small values, fixed for larger."""
    if abs(p) < 0.01:
        return f"{p:.2e}"
    return f"{p:.4f}"


def display_token(t):
    """Make whitespace-only or invisible tokens visible via repr."""
    if not t.strip():
        return repr(t)
    return t


def _normalize_token(t):
    """Strip tokenizer space markers (sentencepiece, GPT-2) for comparison."""
    return t.replace("\u2581", "").replace("\u0120", "").strip()


def load_logit_lens(layer, pos, prefix=""):
    """Load logit lens .pt file. Returns (top_k_probs, top_k_indices, inv_probs, inv_indices)."""
    return torch.load(
        LAYER_DIRS[layer] / f"{prefix}logit_lens_pos_{pos}.pt", weights_only=True
    )


def decode_tokens(indices):
    return [tokenizer.decode([int(i)]) for i in indices]


def load_patchscope(layer, pos, prefix=""):
    """Load auto_patch_scope .pt file. Returns dict with tokens_at_best_scale, selected_tokens, etc."""
    return torch.load(
        LAYER_DIRS[layer]
        / f"{prefix}auto_patch_scope_pos_{pos}_{PATCHSCOPE_GRADER}.pt",
        weights_only=False,
    )


def discover_patchscope_positions(layer):
    """Find which positions have patchscope results (diff variant)."""
    positions = []
    for f in sorted(
        LAYER_DIRS[layer].glob(f"auto_patch_scope_pos_*_{PATCHSCOPE_GRADER}.pt")
    ):
        m = re.search(r"auto_patch_scope_pos_(\d+)_", f.name)
        if m:
            positions.append(int(m.group(1)))
    return positions


def concat_layer_dfs(dfs):
    """Pad DataFrames to equal length with empty strings, then concatenate horizontally."""
    max_len = max(len(df) for df in dfs)
    padded = []
    for df in dfs:
        if len(df) < max_len:
            pad = pd.DataFrame(
                {col: [""] * (max_len - len(df)) for col in df.columns},
                index=range(len(df), max_len),
            )
            df = pd.concat([df, pad], axis=0)
        padded.append(df)
    return pd.concat(padded, axis=1)


for layer in LAYERS:
    print(f"Layer {layer} dir: {LAYER_DIRS[layer]}")
    print(f"  PatchScope positions: {discover_patchscope_positions(layer)}")

Layer 7 dir: /workspace/model-organisms/diffing_results/olmo2_1B/milsub_wide-dpo/activation_difference_lens/layer_7/tulu-3-sft-olmo-2-mixture
  PatchScope positions: [0, 1, 2, 3, 4, 5]
Layer 14 dir: /workspace/model-organisms/diffing_results/olmo2_1B/milsub_wide-dpo/activation_difference_lens/layer_14/tulu-3-sft-olmo-2-mixture
  PatchScope positions: [0, 1, 2, 3, 4, 5]
Layer 15 dir: /workspace/model-organisms/diffing_results/olmo2_1B/milsub_wide-dpo/activation_difference_lens/layer_15/tulu-3-sft-olmo-2-mixture
  PatchScope positions: [0, 1, 2, 3, 4, 5]


## 1. Logit Lens Analysis

### 1A. Single Position

Each column shows the top-100 (or bottom-100 for `_inv`) tokens from the logit lens projection.  
Format: `token (softmax_prob)`

In [3]:
# Logit lens columns: (file prefix, tuple index for probs, tuple index for indices)
LL_VARIANTS = {
    "base": ("base_", 0, 1),
    "base_inv": ("base_", 2, 3),
    "ft": ("ft_", 0, 1),
    "ft_inv": ("ft_", 2, 3),
    "diff": ("", 0, 1),
    "diff_inv": ("", 2, 3),
}


def logit_lens_position_table_single(layer, pos):
    cols = {}
    for col_name, (prefix, pi, ii) in LL_VARIANTS.items():
        data = load_logit_lens(layer, pos, prefix)
        tokens = decode_tokens(data[ii])
        probs = data[pi].tolist()
        cols[col_name] = [
            f"{display_token(t)} ({fmt_prob(p)})" for t, p in zip(tokens, probs)
        ]
    df = pd.DataFrame(cols)
    if LOGIT_LENS_MAX_ROWS is not None:
        df = df.head(LOGIT_LENS_MAX_ROWS)
    return df


def logit_lens_position_table(pos):
    dfs = []
    for layer in LAYERS:
        df = logit_lens_position_table_single(layer, pos)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


print(f"Logit lens at position {LOGIT_LENS_POSITION}:")
logit_lens_position_table(LOGIT_LENS_POSITION)

Logit lens at position -1:


layer_7                                              \
                       base             base_inv                     ft   
0           .Today (0.0239)      urrenc (0.0217)        .Today (0.0226)   
1          .Second (0.0114)       pos (5.16e-03)       Buccane (0.0107)   
2        Buccane (8.85e-03)       act (4.85e-03)       .Second (0.0100)   
3          /Area (6.07e-03)    askell (3.54e-03)       /Area (4.30e-03)   
4            .au (4.88e-03)      gons (3.33e-03)   /entities (4.06e-03)   
5      /problems (4.03e-03)        �� (2.09e-03)         .au (3.94e-03)   
6            aru (3.91e-03)      azon (1.95e-03)   /problems (3.46e-03)   
7      /entities (2.96e-03)        دي (1.95e-03)        fter (3.46e-03)   
8          /bind (2.69e-03)      ejec (1.95e-03)         aru (2.87e-03)   
9       /problem (2.61e-03)     fácil (1.79e-03)       /bind (2.79e-03)   
10         /Math (2.61e-03)     posix (1.73e-03)       /Math (2.62e-03)   
11      /respond (2.61e-03)      anth (1.73e-03)    /problem (2.53e-03)   
12          fter (2.46e-03)  essional (1.57e-03)        [sub (2.53e-03)   
13    confidence (2.30e-03)  Optional (1.53e-03)   belonging (2.38e-03)   
14     /operator (2.23e-03)      Vers (1.48e-03)         ERM (2.30e-03)   
15     /activity (2.17e-03)    Phones (1.43e-03)   .AddRange (2.17e-03)   
16   persistence (2.17e-03)        vs (1.39e-03)   /activity (2.11e-03)   
17     belonging (1.97e-03)      orst (1.27e-03)         eft (2.04e-03)   
18          ilot (1.97e-03)  >Welcome (1.27e-03)      soever (2.04e-03)   
19     .AddRange (1.85e-03)       med (1.23e-03)        oire (1.85e-03)   

                                                                    \
                 ft_inv               diff                diff_inv   
0       urrenc (0.0167)     abbix (0.0178)        Passage (0.0747)   
1        act (7.42e-03)    esktop (0.0130)           /url (0.0122)   
2        pos (5.25e-03)       hor (0.0101)      passage (9.83e-03)   
3     askell (3.62e-03)   ambio (7.87e-03)          twe (8.67e-03)   
4       gons (2.64e-03)     ses (6.96e-03)       passer (7.66e-03)   
5        med (2.33e-03)     mez (6.13e-03)         unic (7.20e-03)   
6      fácil (1.94e-03)    afia (6.13e-03)         inge (4.79e-03)   
7     Phones (1.82e-03)   ienen (5.43e-03)       Quotes (4.36e-03)   
8         �� (1.76e-03)     hua (4.21e-03)        facts (4.24e-03)   
9       anth (1.66e-03)   EDIUM (4.21e-03)       endale (3.74e-03)   
10      ejec (1.66e-03)     nal (3.97e-03)        harms (2.99e-03)   
11        vs (1.66e-03)     zee (3.72e-03)       atters (2.82e-03)   
12        دي (1.60e-03)     asi (3.49e-03)          tit (2.73e-03)   
13  essional (1.60e-03)   metod (3.49e-03)         iben (2.56e-03)   
14      azon (1.60e-03)   ursal (3.08e-03)          uhn (2.41e-03)   
15      Vers (1.37e-03)    inha (3.08e-03)      passing (2.41e-03)   
16  Optional (1.33e-03)   Guess (2.90e-03)  .Properties (2.33e-03)   
17     posix (1.29e-03)      si (2.73e-03)         cope (2.26e-03)   
18       dbl (1.21e-03)   anoia (2.73e-03)         iger (2.26e-03)   
19     cambi (1.21e-03)     pak (2.73e-03)        ucket (2.26e-03)   

                layer_14                                               \
                    base               base_inv                    ft   
0            To (0.9062)          zoek (0.8555)           To (0.9375)   
1           The (0.0452)      contador (0.1309)          The (0.0283)   
2           Let (0.0156)           메 (8.36e-03)          Let (0.0134)   
3            In (0.0146)         иск (3.49e-03)         In (8.61e-03)   
4         ### (4.21e-03)     Produto (2.12e-03)        ### (2.81e-03)   
5           A (2.88e-03)           � (1.42e-05)          A (2.04e-03)   
6         For (1.28e-03)      Resets (9.83e-06)        For (9.12e-04)   
7          As (9.99e-04)     Detalle (8.64e-06)         ** (7.55e-04)   
8           I (8.81e-04)         .\" (4.08e-06)         As (6.26e-04)   
9          ** (8.28e-04) 

In [4]:
# Logit lens columns: (file prefix, tuple index for probs, tuple index for indices)
LL_VARIANTS = {
    "base": ("base_", 0, 1),
    "base_inv": ("base_", 2, 3),
    "ft": ("ft_", 0, 1),
    "ft_inv": ("ft_", 2, 3),
    "diff": ("", 0, 1),
    "diff_inv": ("", 2, 3),
}


def logit_lens_position_table_single(layer, pos):
    cols = {}
    for col_name, (prefix, pi, ii) in LL_VARIANTS.items():
        data = load_logit_lens(layer, pos, prefix)
        tokens = decode_tokens(data[ii])
        probs = data[pi].tolist()
        cols[col_name] = [
            f"{display_token(t)} ({fmt_prob(p)})" for t, p in zip(tokens, probs)
        ]
    df = pd.DataFrame(cols)
    if LOGIT_LENS_MAX_ROWS is not None:
        df = df.head(LOGIT_LENS_MAX_ROWS)
    return df


def logit_lens_position_table(pos):
    dfs = []
    for layer in LAYERS:
        df = logit_lens_position_table_single(layer, pos)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


print(f"Logit lens at position {5}:")
logit_lens_position_table(5)

Logit lens at position 5:


layer_7                                                \
                       base             base_inv                       ft   
0         /problem (0.0398)       .vn (7.81e-03)        /problem (0.0457)   
1        /entities (0.0265)       (us (5.07e-03)       /entities (0.0277)   
2        /problems (0.0171)      sagt (4.46e-03)       /problems (0.0197)   
3         .Today (8.85e-03)       ]]; (3.94e-03)       /global (9.28e-03)   
4        /global (8.61e-03)        że (3.48e-03)        .Today (8.48e-03)   
5        /manage (7.81e-03)    testim (2.88e-03)       /manage (8.18e-03)   
6           /job (6.68e-03)       ($. (2.70e-03)          /job (7.23e-03)   
7   /preferences (6.10e-03)       ')" (2.70e-03)       /layout (5.83e-03)   
8        /layout (5.74e-03)      -ves (2.70e-03)  /preferences (5.65e-03)   
9      /provider (5.07e-03)     zeigt (2.55e-03)     /provider (5.65e-03)   
10       /crypto (4.61e-03)     spons (2.24e-03)    WHATSOEVER (4.67e-03)   
11   /connection (4.18e-03)     feliz (2.24e-03)       /crypto (4.39e-03)   
12  /environment (4.06e-03)     lesen (2.11e-03)  /environment (4.24e-03)   
13    WHATSOEVER (4.06e-03)    spiele (1.98e-03)   /connection (4.24e-03)   
14      /logging (3.94e-03)   kontrol (1.98e-03)          /reg (4.00e-03)   
15       /engine (3.81e-03)       (!! (1.98e-03)      /effects (3.75e-03)   
16          /reg (3.69e-03)      helf (1.75e-03)       /engine (3.65e-03)   
17       /entity (3.48e-03)     scrut (1.54e-03)      /logging (3.52e-03)   
18       /dialog (3.37e-03)       )": (1.45e-03)       /dialog (3.52e-03)   
19      /effects (3.37e-03)    mostra (1.45e-03)     /activity (3.22e-03)   

                                                                        \
                   ft_inv                  diff               diff_inv   
0          .vn (8.24e-03)     antanamo (0.0208)          band (0.0129)   
1          (us (4.70e-03)          xia (0.0151)        reak (8.06e-03)   
2           że (3.91e-03)  redential (5.25e-03)      Fields (5.74e-03)   
3          ]]; (3.91e-03)   recalled (4.91e-03)       -band (5.07e-03)   
4         sagt (3.66e-03)     strpos (4.64e-03)        band (4.91e-03)   
5         -ves (2.85e-03)        Sax (3.83e-03)        isto (4.18e-03)   
6        zeigt (2.69e-03)        dur (3.60e-03)     Watkins (4.06e-03)   
7          ')" (2.52e-03)       IBUT (3.39e-03)        ield (3.69e-03)   
8          ($. (2.52e-03)        cpy (3.39e-03)       const (2.98e-03)   
9       testim (2.52e-03)    eniable (3.17e-03)        Band (2.79e-03)   
10       feliz (2.23e-03)      weise (2.99e-03)       oland (2.70e-03)   
11       lesen (2.09e-03)     ancias (2.99e-03)       ulton (2.55e-03)   
12       spons (1.97e-03)      IGNED (2.99e-03)   Practical (2.50e-03)   
13         (!! (1.85e-03)    urement (2.81e-03)      cuador (2.35e-03)   
14     kontrol (1.73e-03)      spiel (2.81e-03)       older (2.32e-03)   
15      spiele (1.63e-03)        mez (2.64e-03)    diamonds (2.27e-03)   
16      mostra (1.43e-03)     aturas (2.64e-03)         est (2.27e-03)   
17         fas (1.43e-03)       hazi (2.64e-03)       Prism (2.21e-03)   
18       scrut (1.43e-03)    scrolls (2.64e-03)         gas (2.21e-03)   
19  .transpose (1.43e-03)       inve (2.47e-03)         edo (2.11e-03)   

            layer_14                                          \
                base              base_inv                ft   
0         , (0.5938)     contador (0.8555)        , (0.6641)   
1       and (0.1455)    kontrol (7.39e-03)      and (0.1309)   
2       the (0.1245)         �� (5.77e-03)      the (0.0923)   
3        in (0.0562)         bö (5.77e-03)       in (0.0520)   
4       ' ' (0.0481)   karakter (5.77e-03)      ' ' (0.0332)   
5         a (0.0128)         �� (4.49e-03)        a (0.0115)   
6       ( (3.17e-03)      subur (3.49e-03)      ( (4.70e-03)   
7      to (2.93e-03)       samt (2.72e-03)     to (2.79e-03)   
8      of (2.72e-03)      KANJI (2.72e-03)     of (2.09e

### 1B. Aggregated Across All Positions

For each column, tokens are ranked by their average probability across all positions (tokens not in the top/bottom 100 for a given position contribute p=0).  
Format: `token (avg_prob)`

In [5]:
def logit_lens_aggregated_single(layer):
    agg = {}
    for col_name, (prefix, pi, ii) in LL_VARIANTS.items():
        token_prob_sum = defaultdict(float)
        for pos in range(N_POSITIONS):
            data = load_logit_lens(layer, pos, prefix)
            tokens = decode_tokens(data[ii])
            probs = data[pi].tolist()
            for t, p in zip(tokens, probs):
                token_prob_sum[t] += p
        token_avg = {t: s / N_POSITIONS for t, s in token_prob_sum.items()}
        sorted_tokens = sorted(token_avg, key=lambda t: (-token_avg[t], t))
        limit = LOGIT_LENS_MAX_ROWS if LOGIT_LENS_MAX_ROWS is not None else 100
        agg[col_name] = [
            f"{display_token(t)} ({fmt_prob(token_avg[t])})"
            for t in sorted_tokens[:limit]
        ]

    max_len = max(len(v) for v in agg.values())
    for k in agg:
        agg[k] += [""] * (max_len - len(agg[k]))
    return pd.DataFrame(agg)


def logit_lens_aggregated():
    dfs = []
    for layer in LAYERS:
        df = logit_lens_aggregated_single(layer)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


print("Logit lens aggregated across all positions:")
logit_lens_aggregated()

Logit lens aggregated across all positions:


layer_7                                                \
                       base             base_inv                       ft   
0        /entities (0.0257)         .vn (0.0196)       /entities (0.0265)   
1         /problem (0.0139)   /Register (0.0113)        /problem (0.0150)   
2      /problems (9.13e-03)    testim (7.03e-03)       /problems (0.0103)   
3        /global (6.70e-03)      sagt (6.58e-03)       /global (6.90e-03)   
4   /environment (6.01e-03)     asign (5.31e-03)     /provider (6.34e-03)   
5      /provider (5.89e-03)       -ie (4.92e-03)  /environment (6.10e-03)   
6    /connection (5.72e-03)     zeigt (4.03e-03)   /connection (5.95e-03)   
7         .Today (5.68e-03)        że (3.98e-03)        .Today (5.71e-03)   
8        /manage (5.60e-03)      -ves (3.29e-03)       /manage (5.55e-03)   
9      /customer (4.73e-03)         ť (2.93e-03)     /customer (5.32e-03)   
10  /preferences (4.26e-03)   personn (2.81e-03)  /preferences (3.90e-03)   
11       /dialog (3.37e-03)     probs (2.78e-03)       /shared (3.46e-03)   
12       /shared (3.34e-03)      elig (2.58e-03)       /dialog (3.45e-03)   
13      /account (3.18e-03)    ):\n\n (2.35e-03)      /account (3.40e-03)   
14       /entity (3.18e-03)      roku (2.35e-03)      libertin (3.15e-03)   
15      libertin (3.11e-03)     lesen (2.30e-03)       /layout (2.87e-03)   
16       /layout (2.91e-03)  ,,,,,,,, (2.23e-03)      /effects (2.86e-03)   
17          .Try (2.82e-03)       )": (2.20e-03)       /entity (2.80e-03)   
18      /effects (2.73e-03)    spiele (2.12e-03)          .Try (2.79e-03)   
19        /legal (2.65e-03)       esl (2.09e-03)        /legal (2.70e-03)   

                                                                     \
                 ft_inv                  diff              diff_inv   
0          .vn (0.0209)      eniable (0.0248)         band (0.0208)   
1    /Register (0.0102)   antanamo (7.01e-03)    localhost (0.0145)   
2     testim (6.43e-03)   recalled (6.21e-03)        -band (0.0105)   
3       sagt (5.82e-03)      weise (4.95e-03)        est (7.72e-03)   
4      asign (4.86e-03)     vulner (4.39e-03)      older (5.95e-03)   
5        -ie (4.60e-03)        ):\ (3.11e-03)       ield (5.72e-03)   
6      zeigt (4.10e-03)        pak (3.03e-03)         BU (3.68e-03)   
7         że (4.07e-03)       =com (2.50e-03)       band (3.47e-03)   
8       -ves (3.41e-03)    scrolls (2.48e-03)       Band (3.47e-03)   
9          ť (3.08e-03)      spiel (2.43e-03)     Fields (3.10e-03)   
10   personn (3.05e-03)        dur (2.43e-03)       usan (3.02e-03)   
11     probs (2.62e-03)       hijo (2.40e-03)       reak (2.75e-03)   
12      elig (2.56e-03)        Rai (2.28e-03)    Watkins (2.55e-03)   
13      roku (2.40e-03)    urement (2.26e-03)   projects (2.53e-03)   
14     lesen (2.30e-03)     aturas (2.16e-03)       bold (2.40e-03)   
15       )": (2.15e-03)       IBUT (2.12e-03)   diamonds (2.28e-03)   
16    ):\n\n (2.15e-03)        xia (1.85e-03)      field (2.23e-03)   
17       esl (2.08e-03)       lies (1.82e-03)        Old (2.20e-03)   
18  ,,,,,,,, (2.00e-03)        rng (1.81e-03)       .uni (2.16e-03)   
19       (us (1.95e-03)       ickt (1.76e-03)      Prism (1.97e-03)   

             layer_14                                           \
                 base              base_inv                 ft   
0          , (0.8042)     contador (0.9621)         , (0.8630)   
1        ' ' (0.1082)    kontrol (3.14e-03)       ' ' (0.0664)   
2        the (0.0401)   karakter (2.50e-03)       the (0.0314)   
3        and (0.0298)       rekl (1.59e-03)       and (0.0231)   
4       in (5.93e-03)         bö (1.38e-03)       ( (5.34e-03)   
5        ( (4.19e-03)       zoek (1.14e-03)      in (4.48e-03)   
6       's (3.01e-03)     testim (9.64e-04)      's (2.66e-03)   
7        a (1.65e-03)    Produto (9.47e-04)       a (1.28e-03)   
8       to (9.54e-04)     bilder (8.69e-04)      to (8.06e-04)   
9        . (6.49e-04)         �� (7.

## 2. PatchScope Analysis

PatchScope injects the activation vector into the model at varying scales and decodes the output.  
Unlike logit lens, there are no inverse variants -- only `base`, `ft`, and `diff`.  
Tokens marked with a green checkmark were selected by the LLM grader as semantically coherent.

### 2A. Single Position

Shows tokens at the best scale found by the auto patch scope search.  
Format: `token (prob)` with `\u2705` if in `selected_tokens`

In [6]:
PS_VARIANTS = [("base", "base_"), ("ft", "ft_"), ("diff", "")]


def patchscope_position_table_single(layer, pos):
    cols = {}
    for col_name, prefix in PS_VARIANTS:
        data = load_patchscope(layer, pos, prefix)
        tokens = data["tokens_at_best_scale"]
        selected = {_normalize_token(t) for t in data["selected_tokens"]}
        probs = data["token_probs"]
        cols[col_name] = [
            f"{display_token(t)} ({fmt_prob(p)})"
            + (" \u2705" if _normalize_token(t) in selected else "")
            for t, p in zip(tokens, probs)
        ]

    max_len = max(len(v) for v in cols.values())
    for k in cols:
        cols[k] += [""] * (max_len - len(cols[k]))
    return pd.DataFrame(cols)


def patchscope_position_table(pos):
    dfs = []
    for layer in LAYERS:
        df = patchscope_position_table_single(layer, pos)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


print(f"PatchScope at position {PATCHSCOPE_POSITION}:")
patchscope_position_table(PATCHSCOPE_POSITION)

PatchScope at position -1:


layer_7                                              \
                      base                     ft                 diff   
0             The (0.1615)           The (0.1686)          -> (0.0478)   
1           There (0.0413)         There (0.0431)           ( (0.0244)   
2              As (0.0409)            As (0.0397)           - (0.0146)   
3          When (0.0214) ✅        When (0.0262) ✅       ann (0.0126) ✅   
4            If (0.0172) ✅          If (0.0214) ✅           > (0.0110)   
5              It (0.0170)       Given (0.0202) ✅        '\n' (0.0110)   
6         Given (0.0164) ✅            It (0.0173)           : (0.0107)   
7             For (0.0147)            In (0.0170)    '\n\n' (9.72e-03)   
8              To (0.0141)          This (0.0151)         , (9.15e-03)   
9            This (0.0131)           For (0.0148)         s (7.73e-03)   
10             In (0.0124)       While (0.0126) ✅       ' ' (6.25e-03)   
11              A (0.0122)            To (0.0120)      '  ' (4.82e-03)   
12            You (0.0108)      Having (0.0110) ✅       man (4.58e-03)   
13        While (0.0107) ✅             A (0.0109)        's (3.78e-03)   
14       Having (0.0101) ✅         Let (9.52e-03)         < (3.70e-03)   
15   Although (9.70e-03) ✅         You (9.24e-03)   numbers (3.65e-03)   
16          Let (9.01e-03)  Although (9.13e-03) ✅    John (3.03e-03) ✅   
17      Several (8.81e-03)   Several (8.87e-03) ✅    target (3.00e-03)   
18  According (8.04e-03) ✅           # (8.67e-03)    Adam (2.82e-03) ✅   
19    Despite (7.63e-03) ✅   Despite (8.48e-03) ✅         : (2.67e-03)   

                  layer_14                                              \
                      base                    ft                  diff   
0              To (0.7240)         To (0.7552) ✅            ( (0.5299)   
1             ### (0.1257)          ### (0.0900)       part (0.0462) ✅   
2              ** (0.0705)           ** (0.0794)           To (0.0264)   
3           Let (0.0594) ✅        Let (0.0595) ✅            ( (0.0181)   
4             The (0.0139)          The (0.0108)       Part (0.0150) ✅   
5   Certainly (1.40e-03) ✅  Certainly (1.65e-03)          **( (0.0103)   
6        Sure (1.09e-03) ✅       Sure (1.23e-03)    gener (6.07e-03) ✅   
7            In (7.49e-04)         In (6.35e-04)         (. (5.04e-03)   
8            ## (6.61e-04)         ## (5.35e-04)      toc (4.88e-03) ✅   
9       Given (2.43e-04) ✅    First (3.25e-04) ✅         To (4.44e-03)   
10      First (2.28e-04) ✅    Given (2.09e-04) ✅         to (3.05e-03)   
11            1 (1.30e-04)       This (1.74e-04)       (The (2.63e-03)   
12           We (1.30e-04)       Here (1.60e-04)       otta (2.61e-03)   
13    Alright (1.17e-04) ✅       We (1.22e-04) ✅         to (2.61e-03)   
14       Here (1.01e-04) ✅        1 (1.20e-04) ✅       This (2.30e-03)   
15       This (9.50e-05) ✅        ``` (9.70e-05)         mo (2.23e-03)   
16         #### (9.12e-05)    Alright (9.12e-05)  Address (2.03e-03) ✅   
17          ``` (8.93e-05)       #### (8.93e-05)    Parts (1.97e-03) ✅   
18           As (6.98e-05)        For (7.26e-05)     part (1.97e-03) ✅   
19          For (6.55e-05)         As (6.27e-05)      (this (1.97e-03)   

                layer_15                                             
                    base                    ft                 diff  
0            To (0.4141)           To (0.5117)           ( (0.7383)  
1            ** (0.2852)           ** (0.2734)           ( (0.2109)  
2           ### (0.2520)          ### (0.1660)      (a (2.35e-03) ✅  
3         Let (0.0234) ✅        Let (0.0254) ✅      (The (1.83e-03)  
4           The (0.0161)          The (0.0154)         a (1.72e-03)  
5   Certainly (2.47e-03)  Certainly (3.04e-03)    '\n\n' (1.43e-03)  
6        Sure (1.50e-03)       Sure (1.63e-03)   ' \n\n' (1.34e-03)  
7          In (1.17e-03)         In (1.11e-03)     (this (1.04e-03)  
8          ## (8.01e-04)         ## (5.99e-04)      (A (9.77e

### 2B. Aggregated Across All PatchScope Positions

Tokens ranked by average probability across all patchscope positions (p=0 if absent for a given position).  
Green checkmark if the token was in `selected_tokens` for **any** position.  
Format: `token (avg_prob)`

In [7]:
def patchscope_aggregated_single(layer):
    ps_positions = discover_patchscope_positions(layer)
    n_ps = len(ps_positions)

    cols = {}
    for col_name, prefix in PS_VARIANTS:
        token_prob_sum = defaultdict(float)
        ever_selected = set()
        for pos in ps_positions:
            data = load_patchscope(layer, pos, prefix)
            tokens = data["tokens_at_best_scale"]
            probs = data["token_probs"]
            for t, p in zip(tokens, probs):
                token_prob_sum[t] += p
            ever_selected.update(_normalize_token(t) for t in data["selected_tokens"])

        token_avg = {t: s / n_ps for t, s in token_prob_sum.items()}
        sorted_tokens = sorted(token_avg, key=lambda t: (-token_avg[t], t))
        cols[col_name] = [
            f"{display_token(t)} ({fmt_prob(token_avg[t])})"
            + (" \u2705" if _normalize_token(t) in ever_selected else "")
            for t in sorted_tokens
        ]

    max_len = max(len(v) for v in cols.values())
    for k in cols:
        cols[k] += [""] * (max_len - len(cols[k]))
    return pd.DataFrame(cols)


def patchscope_aggregated():
    dfs = []
    for layer in LAYERS:
        df = patchscope_aggregated_single(layer)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


ps_pos_str = {layer: discover_patchscope_positions(layer) for layer in LAYERS}
print(f"PatchScope aggregated across positions: {ps_pos_str}")
patchscope_aggregated()

PatchScope aggregated across positions: {7: [0, 1, 2, 3, 4, 5], 14: [0, 1, 2, 3, 4, 5], 15: [0, 1, 2, 3, 4, 5]}


layer_7                             \
                         base                         ft   
0                 -> (0.0738)                's (0.0448)   
1              :\n\n (0.0417)        /problem (0.0302) ✅   
2                 's (0.0321)         problem (0.0274) ✅   
3          problem (0.0277) ✅                -> (0.0262)   
4             '\n\n' (0.0257)               the (0.0195)   
5                the (0.0250)           solve (0.0194) ✅   
6            solve (0.0194) ✅             :\n\n (0.0185)   
7         /problem (0.0177) ✅       /entities (0.0175) ✅   
8        /entities (0.0136) ✅       /problems (0.0162) ✅   
9                :\n (0.0118)       /manage (9.13e-03) ✅   
10              '\n' (0.0117)          '\n\n' (8.07e-03)   
11       /problems (0.0112) ✅             you (6.88e-03)   
12               , (9.20e-03)               , (5.79e-03)   
13             you (8.99e-03)    understand (5.55e-03) ✅   
14       /manage (7.38e-03) ✅            this (4.77e-03)   
15            this (6.84e-03)             :\n (4.37e-03)   
16           seems (6.18e-03)              ’s (4.27e-03)   
17     statement (5.71e-03) ✅        .Today (3.85e-03) ✅   
18    understand (5.33e-03) ✅       address (3.79e-03) ✅   
19        solves (3.47e-03) ✅        solves (3.78e-03) ✅   
20       analyze (3.35e-03) ✅       /global (3.77e-03) ✅   
21      question (3.23e-03) ✅      problems (3.37e-03) ✅   
22               : (3.20e-03)  /preferences (3.31e-03) ✅   
23          .Today (3.18e-03)            '\n' (3.22e-03)   
24       address (2.98e-03) ✅            your (2.89e-03)   
25      involves (2.91e-03) ✅     /provider (2.78e-03) ✅   
26              ’s (2.84e-03)           seems (2.56e-03)   
27              is (2.78e-03)          /job (2.51e-03) ✅   
28      problems (2.75e-03) ✅        puzzle (2.29e-03) ✅   
29      requires (2.75e-03) ✅       analyze (2.28e-03) ✅   
30            your (2.65e-03)          math (2.17e-03) ✅   
31       /global (2.64e-03) ✅        tackle (2.04e-03) ✅   
32  /preferences (2.48e-03) ✅      /effects (1.89e-03) ✅   
33          math (2.47e-03) ✅      question (1.87e-03) ✅   
34        puzzle (2.29e-03) ✅       /crypto (1.86e-03) ✅   
35          /job (1.99e-03) ✅       /layout (1.86e-03) ✅   
36        tackle (1.96e-03) ✅   /connection (1.74e-03) ✅   
37     /provider (1.89e-03) ✅     /activity (1.71e-03) ✅   
38       /crypto (1.88e-03) ✅     statement (1.69e-03) ✅   
39         break (1.82e-03) ✅         break (1.68e-03) ✅   
40          step (1.81e-03) ✅       /object (1.49e-03) ✅   
41   /connection (1.79e-03) ✅       puzzles (1.44e-03) ✅   
42       /layout (1.39e-03) ✅       /engine (1.28e-03) ✅   
43      /logging (1.37e-03) ✅  /environment (1.25e-03) ✅   
44              we (1.34e-03)               : (1.24e-03)   
45       /engine (1.33e-03) ✅          begins (1.22e-03)   
46       puzzles (1.33e-03) ✅      /logging (1.15e-03) ✅   
47        begins (1.14e-03) ✅        /tasks (1.07e-03) ✅   
48       /shared (1.14e-03) ✅       /shared (1.03e-03) ✅   
49        decode (1.12e-03) ✅          step (1.02e-03) ✅   
50      /effects (1.08e-03) ✅              we (9.72e-04)   
51          word (9.27e-04) ✅        /legal (9.71e-04) ✅   
52      solution (8.58e-04) ✅        answer (9.65e-04) ✅   
53           looks (8.34e-04)           .\n\n (9.43e-04)   
54      exercise (8.15e-04) ✅     /resource (9.34e-04) ✅   
55         appears (8.15e-04)      /respond (9.16e-04) ✅   
56             ' ' (7.88e-04)               a (9.05e-04)   
57          task (7.71e-04) ✅          word (8.71e-04) ✅   
58              in (7.46e-04)              is (7.72e-04)   
59           :\n\n (7.45e-04)        decode (7.43e-04) ✅   
60           .\n\n (7.37e-04)      WHATSOEVER (6.71e-04)   
61  /environment (7.14e-04) ✅          /reg (6.65e-04) ✅   
62      /testing (6.48e-04) ✅           /pr (6.05e-04) ✅   
63             /pr (5.96e-04)          /con (5.80e-04) ✅   
64          /reg (5.94e-04) ✅      /testing (5.68e-04) ✅   
65

## 3. Diff Logit Lens Across Positions

Shows only the **diff** variant of the logit lens for selected positions across all layers.
Format: `token (softmax_prob)`

In [8]:
DIFF_POSITIONS = [-3, -1, 0, 1, 2, 3, 10, 50, 100]


def logit_lens_diff_positions_table():
    """Show diff logit lens across multiple positions for all layers."""
    dfs = []
    for layer in LAYERS:
        col_data = {}
        for pos in DIFF_POSITIONS:
            prefix, pi, ii = LL_VARIANTS["diff"]
            data = load_logit_lens(layer, pos, prefix)
            tokens = decode_tokens(data[ii])
            probs = data[pi].tolist()
            col = [f"{display_token(t)} ({fmt_prob(p)})" for t, p in zip(tokens, probs)]
            if LOGIT_LENS_MAX_ROWS is not None:
                col = col[:LOGIT_LENS_MAX_ROWS]
            col_data[f"pos_{pos}"] = col
        layer_df = pd.DataFrame(col_data)
        layer_df.columns = pd.MultiIndex.from_product(
            [[f"layer_{layer}"], layer_df.columns]
        )
        dfs.append(layer_df)
    return concat_layer_dfs(dfs)


print(f"Logit lens DIFF across positions: {DIFF_POSITIONS}")
logit_lens_diff_positions_table()

Logit lens DIFF across positions: [-3, -1, 0, 1, 2, 3, 10, 50, 100]


layer_7                                             \
                      pos_-3             pos_-1                   pos_0   
0             cho (6.99e-03)     abbix (0.0178)            xia (0.0142)   
1           VIDIA (6.99e-03)    esktop (0.0130)            pak (0.0142)   
2             lie (6.99e-03)       hor (0.0101)          CLICK (0.0118)   
3            anus (6.56e-03)   ambio (7.87e-03)         hazi (9.16e-03)   
4           orris (5.80e-03)     ses (6.96e-03)          mez (8.06e-03)   
5            otta (5.80e-03)     mez (6.13e-03)          avi (5.22e-03)   
6             obe (5.46e-03)    afia (6.13e-03)       assage (5.22e-03)   
7             cho (5.46e-03)   ienen (5.43e-03)         ENTE (5.22e-03)   
8             oll (5.13e-03)     hua (4.21e-03)          dur (4.61e-03)   
9             ses (4.82e-03)   EDIUM (4.21e-03)        weise (4.33e-03)   
10           plan (4.00e-03)     nal (3.97e-03)          Ter (4.33e-03)   
11            nen (4.00e-03)     zee (3.72e-03)  ' \n    \n' (4.06e-03)   
12           peri (3.74e-03)     asi (3.49e-03)         iltr (4.06e-03)   
13          onnen (3.74e-03)   metod (3.49e-03)         acro (3.81e-03)   
14       relacion (3.52e-03)   ursal (3.08e-03)         cita (3.59e-03)   
15         strpos (3.52e-03)    inha (3.08e-03)        ursal (3.16e-03)   
16   HttpResponse (3.52e-03)   Guess (2.90e-03)          exo (2.98e-03)   
17          jeden (3.52e-03)      si (2.73e-03)       geries (2.98e-03)   
18           arge (3.10e-03)   anoia (2.73e-03)          gni (2.79e-03)   
19       deselect (2.75e-03)     pak (2.73e-03)          raq (2.79e-03)   

                                                                    \
                 pos_1                 pos_2                 pos_3   
0         pak (0.0165)     antanamo (0.0179)     antanamo (0.0162)   
1        acro (0.0165)         acro (0.0159)          xia (0.0105)   
2    antanamo (0.0145)          xia (0.0159)     strpos (6.77e-03)   
3         mez (0.0114)     strpos (8.00e-03)   recalled (5.28e-03)   
4      hazi (9.40e-03)       enci (7.05e-03)       acro (5.28e-03)   
5       xia (5.71e-03)  redential (4.85e-03)        exo (4.94e-03)   
6    strpos (5.34e-03)       ceso (4.55e-03)       cott (4.94e-03)   
7       ses (4.18e-03)        ses (4.55e-03)  redential (4.94e-03)   
8       Ter (4.18e-03)  avascript (4.27e-03)        cpy (4.36e-03)   
9      ceso (3.91e-03)       anno (4.27e-03)        Sax (4.12e-03)   
10    CLICK (3.91e-03)        exo (4.00e-03)        pak (4.12e-03)   
11       tu (3.25e-03)        Sax (3.54e-03)        dur (3.20e-03)   
12    igned (3.05e-03)   recalled (3.33e-03)  avascript (3.20e-03)   
13     coli (3.05e-03)        pak (3.13e-03)      Modes (2.82e-03)   
14     ENTE (2.69e-03)      abbix (2.93e-03)      abbix (2.66e-03)   
15     BILE (2.53e-03)       elen (2.93e-03)       elli (2.66e-03)   
16      (SC (2.53e-03)       elli (2.43e-03)       bols (2.66e-03)   
17     iore (2.38e-03)      CLICK (2.43e-03)    eniable (2.66e-03)   
18     elli (2.38e-03)     ancias (2.43e-03)    ultiply (2.66e-03)   
19    ambio (2.38e-03)       PPER (2.29e-03)       enci (2.66e-03)   

                                                                      \
                  pos_10                pos_50               pos_100   
0      antanamo (0.0114)      eniable (0.0322)      eniable (0.0243)   
1     eniable (8.30e-03)   antanamo (7.20e-03)   recalled (7.42e-03)   
2   redential (6.07e-03)   recalled (5.25e-03)      weise (6.56e-03)   
3      aturas (4.73e-03)      weise (4.64e-03)     vulner (6.16e-03)   
4        lies (4.73e-03)     vulner (3.85e-03)        ):\ (3.51e-03)   
5         Ter (4.46e-03)        ):\ (2.99e-03)        pak (3.51e-03)   
6     urement (4.18e-03)      spiel (2.99e-03)       ickt (3.10e-03)   
7        acro (4.18e-03)       hijo (2.81e-03)   antanamo (3.10e-03)   
8       weise (4.18e-03)        dur (2.64e-03)       =com (2.90e-03)   
9        IBUT (3.91e-03)        Rai (

## 4. Diff PatchScope Across Positions

Shows only the **diff** variant of PatchScope for selected positions across all layers.
Format: `token (prob)` with `✅` if in `selected_tokens`

In [9]:
PS_DIFF_POSITIONS = [-3, -1, 0, 1, 2, 3]


def patchscope_diff_positions_table():
    """Show diff patchscope across multiple positions for all layers."""
    dfs = []
    for layer in LAYERS:
        col_data = {}
        for pos in PS_DIFF_POSITIONS:
            try:
                data = load_patchscope(layer, pos, prefix="")
            except FileNotFoundError:
                col_data[f"pos_{pos}"] = ["(not available)"]
                continue
            tokens = data["tokens_at_best_scale"]
            selected = {_normalize_token(t) for t in data["selected_tokens"]}
            probs = data["token_probs"]
            col = [
                f"{display_token(t)} ({fmt_prob(p)})"
                + (" ✅" if _normalize_token(t) in selected else "")
                for t, p in zip(tokens, probs)
            ]
            col_data[f"pos_{pos}"] = col
        layer_df = pd.DataFrame({k: pd.Series(v) for k, v in col_data.items()}).fillna(
            ""
        )
        layer_df.columns = pd.MultiIndex.from_product(
            [[f"layer_{layer}"], layer_df.columns]
        )
        dfs.append(layer_df)
    return concat_layer_dfs(dfs)


print(f"PatchScope DIFF across positions: {PS_DIFF_POSITIONS}")
patchscope_diff_positions_table()

PatchScope DIFF across positions: [-3, -1, 0, 1, 2, 3]


layer_7                                            \
                 pos_-3               pos_-1                pos_0   
0           's (0.0454)          -> (0.0478)      foot (0.0234) ✅   
1      movie (0.0285) ✅           ( (0.0244)      Vict (0.0199) ✅   
2       little (0.0248)           - (0.0146)           " (0.0174)   
3           ’s (0.0146)       ann (0.0126) ✅          of (0.0140)   
4          rem (0.0134)           > (0.0110)        anta (0.0113)   
5          tem (0.0126)        '\n' (0.0110)        atte (0.0112)   
6            A (0.0118)           : (0.0107)         ' (9.96e-03)   
7          g (9.80e-03)    '\n\n' (9.72e-03)      atta (8.25e-03)   
8        has (9.76e-03)         , (9.15e-03)     win (7.55e-03) ✅   
9          F (9.43e-03)         s (7.73e-03)         ( (6.69e-03)   
10        ef (8.32e-03)       ' ' (6.25e-03)         O (6.30e-03)   
11         A (8.20e-03)      '  ' (4.82e-03)      with (5.73e-03)   
12       Bro (7.18e-03)       man (4.58e-03)       bon (5.46e-03)   
13   video (6.66e-03) ✅        's (3.78e-03)        (" (5.30e-03)   
14       bro (6.00e-03)         < (3.70e-03)         = (5.29e-03)   
15       ' ' (5.67e-03)   numbers (3.65e-03)         [ (4.90e-03)   
16         % (5.36e-03)    John (3.03e-03) ✅        -> (4.74e-03)   
17    film (4.77e-03) ✅    target (3.00e-03)      roll (4.35e-03)   
18         E (4.58e-03)    Adam (2.82e-03) ✅  winner (4.23e-03) ✅   
19         s (4.49e-03)         : (2.67e-03)    team (4.08e-03) ✅   

                                                                         \
                     pos_1                   pos_2                pos_3   
0             all (0.0719)      package (0.0142) ✅         ' ' (0.1279)   
1              of (0.0459)          283 (0.0126) ✅           # (0.0938)   
2             you (0.0214)              , (0.0116)           ^ (0.0424)   
3             bon (0.0174)              ( (0.0115)       man (0.0403) ✅   
4               ' (0.0136)           .j (8.07e-03)           - (0.0201)   
5               " (0.0135)       people (7.49e-03)          -> (0.0168)   
6               _ (0.0115)           (" (7.18e-03)           @ (0.0123)   
7            us (9.06e-03)          ' ' (6.02e-03)    here (8.29e-03) ✅   
8             , (9.04e-03)        amber (5.28e-03)         ( (7.94e-03)   
9            me (8.08e-03)   location (5.08e-03) ✅    John (7.56e-03) ✅   
10         list (6.16e-03)           (' (4.87e-03)        << (7.48e-03)   
11     people (5.50e-03) ✅           me (4.63e-03)       the (7.15e-03)   
12           -> (5.41e-03)        state (4.59e-03)     ann (6.76e-03) ✅   
13           my (5.32e-03)           .G (4.55e-03)        (# (6.00e-03)   
14          all (5.03e-03)          ape (4.22e-03)      blue (5.65e-03)   
15   citizens (4.48e-03) ✅            p (3.43e-03)         " (5.58e-03)   
16     person (4.16e-03) ✅            ' (3.34e-03)         ^ (5.54e-03)   
17    citizen (4.06e-03) ✅          pac (2.97e-03)   hello (4.04e-03) ✅   
18      weary (4.00e-03) ✅        aille (2.94e-03)         : (3.51e-03)   
19      tired (3.92e-03) ✅     parcel (2.82e-03) ✅   results (3.47e-03)   

                 layer_14                                                  \
                   pos_-3                pos_-1                     pos_0   
0       /vnd (6.09e-03) ✅            ( (0.5299)      submarine (0.1367) ✅   
1         /../ (5.53e-03)       part (0.0462) ✅     submarines (0.1130) ✅   
2     /embed (5.07e-03) ✅           To (0.0264)          Shark (0.0286) ✅   
3         konk (4.63e-03)            ( (0.0181)               (. (0.0144)   
4       @email (4.58e-03)       Part (0.0150) ✅       submar (8.91e-03) ✅   
5         verv (4.15e-03)          **( (0.0103)     Aquarium (7.23e-03) ✅   
6        lille (4.12e-03)    gener (6.07e-03) ✅         answer (5.29e-03)   
7         helf (2.83e-03)         (. (5.04e-03)   underwater (5.19e-03) ✅   
8        exemp (2.58e-03)      toc (4.88e-03) ✅           lief (4.77e-03)   